# 리포트 34 — 스톡 Paths.doppler 로는 블레이드 변조가 안 나온다 — SceneObject.velocity 가 객체당 강체 1벡터다

> ### 한 일
> **설치본 Sionna 의 장면 객체에 속도를 직접 넣고 경로 도플러를 읽어 자유도를 셌다.**

### 결과
1. 장면 객체 하나가 갖는 속도 자유도는 3 개 [^1] 다 — 평행이동 세 성분이고, 회전 자유도는 그 밖이다.
2. 정지 장면(**Mini 2**)에서 도플러가 0 이 아닌 경로는 0 개 [^2] (전체 98 개 [^3]) — 배선 자체는 정상이다.
3. **Matrice 4E** 의 프롭 그룹에 강체속도를 주면 **표적 경유 경로** 296 개 [^4] 가 모두 같은 부호로 몰린다 — 크기는 강체 투영에 따라 0 ~ 187.5 Hz [^5] 로 퍼지고, 그 최대가 강체 예측 181.3 Hz [^6] 와 같은 자리에 선다.
4. 전진날과 후퇴날이 갈리려면 그 둘이 반대 부호를 받아야 하는데, 자유도 3 [^1] 개짜리 벡터 하나가 그것을 표현한다.
5. 그래서 남는 길은 하나다 — 시간표본마다 자세를 새로 놓고 다시 쏘는 것이고, 그 절차가 [편 35 «슬로타임 복소열»](35_md-slowtime.ipynb) 이다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 자유도 세기 | 설치본 `sionna.rt.SceneObject.velocity` 의 성분 수를 부품 객체마다 직접 읽었다 |
| 강체 주입 시험 | 프롭 그룹에만 속도 (0, 0, 30) m/s 를 주고 `Paths.doppler` 의 고유값을 셌다 |
| 대조 | 같은 장면을 정지 상태로 한 번 더 추적해 도플러가 0 인지 확인했다 |
| 무엇을 안 물었나 | Sionna 가 «회전하는 기하의 왕복 위상» 을 따라가는지는 이 시험의 밖이다 — [편 36 «두 엔진»](36_md-two-engines.ipynb) 이 그것을 잰다 |

### 재현

```bash
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/report15_probe.py
```

| | |
|---|---|
| 출력 | `outputs/report15_probe.json` |
| 소요 | 약 7 분 (GPU 1장) |

---

## 무엇을 물었나

Sionna 는 장면 객체에 속도를 주면 경로마다 도플러를 계산해 준다. 그 기능만으로 **블레이드 마이크로도플러**(도는 날개가 만드는 변조)가 나오는지가 첫 갈래다.

나온다면 로터를 돌리는 비싼 길을 건너뛴다. 안 나온다면 시간표본마다 자세를 다시 놓고 광선을 다시 쏘는 길로 간다. 두 길의 비용 차이가 커서 먼저 확인했다.

## 자유도를 세면 답이 정해진다

| 무엇을 | 기체 | 값 |
|---|---|---|
| 객체당 속도 자유도 | Mini 2 | 3 개 [^1] |
| 정지 장면의 0 아닌 도플러 | Mini 2 | 0 개 [^2] |
| 강체속도 주입 시 표적 경유 경로 | Matrice 4E | 296 개 [^4] |
| 그 경로의 도플러 최대 | Matrice 4E | 187.5 Hz [^5] |
| 강체 운동학이 예측하는 값 | Matrice 4E | 181.3 Hz [^6] |

프롭 전체에 강체속도 하나를 주면 표적 경유 경로가 전부 같은 부호로 몰리고, 크기는 각 경로의 강체 투영에 따라 0 부터 표의 최대값까지 퍼진다 — 그 최대값이 강체 예측과 같은 자리에 선다. 블레이드 전진/후퇴가 갈라지려면 부호가 반대인 두 값이 나와야 하는데, 이 표의 도플러는 한 부호로 몰린다.

## 표의 두 줄은 잣대가 다르다

«표적 경유 경로» 는 `outputs/report15_probe.json : airframes.matrice4e.A_doppler.rigid_prop_velocity.n_target_paths` 이고, 그 키는 표적의 어느 부품이든 스친 경로를 전부 센다 — 프롭만 센 수는 따로 세야 한다. 판정은 그대로 선다: 경로 집합이 넓을수록 부호가 갈릴 기회가 많은데도 도플러가 한 부호로 몰린다.

위 두 줄은 **Mini 2**, 아래 세 줄은 **Matrice 4E** 다. 자유도 세기는 두 기체에서 같은 답을 낸다 — `outputs/report15_probe.json : airframes.*.A_doppler.velocity_dof_per_object` 의 모든 부품이 평행이동 세 성분뿐이라, 판정이 기체 선택에 안 걸린다.

## 이것은 구현 문제가 아니라 자료구조 문제다

속도가 객체당 벡터 하나라는 것은 «그 객체가 강체로 평행이동한다» 는 뜻이다. 회전하는 프로펠러는 같은 객체 안에서 점마다 속도가 다르고, 그 차이를 담을 자리가 벡터 하나 밖에 있다.

부품을 날개 하나하나로 쪼개도 마찬가지다 — 날개 하나 안에서도 뿌리와 끝의 속도가 다르고, 그 차이가 바로 날개끝 확산을 만드는 양이다.

판정 문장은 산출물에 그대로 있다 — `outputs/report15_verdict.json : branch1_paths_doppler.answer`.

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 시간표본마다 자세를 새로 놓고 다시 쏜다 | 블레이드 변조를 만드는 유일한 길이 실제로 작동하는지가 갈린다 | [편 35 «슬로타임 복소열»](35_md-slowtime.ipynb) |
| Sionna 가 회전 기하의 왕복 위상을 따라가는지 우리 커널과 맞댄다 | 위상을 광선 엔진에 맡길 수 있는지가 정해진다 | [편 36 «두 엔진»](36_md-two-engines.ipynb) |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 6개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/report15_verdict.json` | `branch1_paths_doppler.evidence.max_dof` | 3 |
| [^2] | `outputs/report15_verdict.json` | `branch1_paths_doppler.evidence.doppler_nonzero_paths_static_scene` | 0 |
| [^3] | `outputs/report15_verdict.json` | `branch1_paths_doppler.evidence.n_paths` | 98 |
| [^4] | `outputs/report15_probe.json` | `airframes.matrice4e.A_doppler.rigid_prop_velocity.n_target_paths` | 296 |
| [^5] | `outputs/report15_probe.json` | `airframes.matrice4e.A_doppler.rigid_prop_velocity.doppler_max_hz` | 187.5 |
| [^6] | `outputs/report15_probe.json` | `airframes.matrice4e.A_doppler.rigid_prop_velocity.predicted_rigid_hz` | 181.3 |